# 03.4 — Machine Learning Tweedie Regression Benchmark (Validation Protocol v1.0.21)

Đánh giá chuẩn hóa 8 kịch bản học máy phân phối Tweedie Compound Poisson-Gamma (**SC-28 đến SC-35**), bao gồm:
- **2 Họ Mô hình**: LightGBM Tweedie (`lightgbm_tweedie`), XGBoost Tweedie (`xgboost_tweedie`).
- **Tham số phương sai (Tweedie Variance Power)**: Khóa cố định **$p = 1.5$** theo Project Decision.
- **Hàm liên kết Log Link**: Bảo đảm đầu ra dự báo nội tại luôn dương nghiêm ngặt (không cần cắt ngưỡng nhân tạo).
- **4 Cấp độ Đặc trưng Bảng ($F_0, F_1, F_2, F_3$)**: 36, 53, 65, và 70 biến.

### Quy tắc giao thức cốt lõi:
1. **Single-Fit trên 6,293 mẫu Dev Train**: Áp dụng common origins mask (loại 29 dòng warmup đầu) để bảo đảm so sánh công bằng tuyệt đối.
2. **Đánh giá trên đúng 724 Validation Origins**: Lăn dự báo đa bước ($H=7$) qua 724 mốc thời gian độc lập (`2018-05-01` đến `2020-04-23`).
3. **Zero Final Test Access Rule**: Không tiếp cận tập dữ liệu Test.

In [1]:
import sys
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

from src.evaluation.protocols import (
    load_canonical_data,
    build_purged_direct_samples,
    compute_benchmark_metrics,
    CANONICAL_DEV_TRAIN_MASE_SCALE,
    ExecutionStatus,
)
from src.evaluation.metrics import evaluate_point_forecast_metrics
from src.featurengineering.builder import FeatureBuilder, FeatureConfig

# Nạp dữ liệu chuẩn hóa (Dev Train và Validation)
train_full, val_set, _ = load_canonical_data()
print(f"Loaded canonical data: Dev Train = {len(train_full):,} ngày, Validation = {len(val_set):,} ngày")


## 2. Chuẩn bị Đặc trưng Bảng ($F_0, F_1, F_2, F_3$) & Ground Truth 724 Origins

In [2]:
num_val_origins = 724
df_eval = pd.concat([train_full, val_set], ignore_index=True)
y_eval = df_eval['Lượng mưa'].astype(float).values
dev_train_len = len(train_full)  # 6329
origin_indices = [dev_train_len + i for i in range(num_val_origins)]
origin_dates = val_set['Ngày'].iloc[:num_val_origins].values

# Ground truth target window (724, 7)
y_val_true = np.array([y_eval[idx + 1 : idx + 8] for idx in origin_indices], dtype=np.float64)
assert y_val_true.shape == (num_val_origins, 7)

feature_data = {}
expected_counts = {0: 36, 1: 53, 2: 65, 3: 70}

for lvl in range(4):
    cfg = FeatureConfig.from_level(lvl)
    fb = FeatureBuilder(cfg).fit(train_full)
    
    df_feat_train = fb.transform(train_full, drop_warmup=False)
    feat_cols = [c for c in df_feat_train.columns if c not in ['Ngày', 'Lượng mưa']]
    assert len(feat_cols) == expected_counts[lvl]
    
    # Common training mask (6,293 mẫu)
    mask = np.ones(len(train_full) - 7, dtype=bool)
    mask[:29] = False
    
    X_train, Y_train, _ = build_purged_direct_samples(
        df_feat_train, feature_cols=feat_cols, target_col='Lượng mưa', horizon=7, origin_mask=mask
    )
    assert len(X_train) == 6293
    
    # Validation features Mode B
    fset_val = fb.build_feature_set(val_set, context_df=train_full)
    X_val = fset_val.X.iloc[:num_val_origins].values.astype(np.float64)
    assert X_val.shape == (num_val_origins, expected_counts[lvl])
    
    feature_data[f"F{lvl}"] = {
        "feature_cols": feat_cols,
        "feature_count": expected_counts[lvl],
        "X_train": X_train,
        "Y_train": Y_train,
        "X_val": X_val,
    }
    print(f"[F{lvl}] Features: {expected_counts[lvl]} cols | X_train: {X_train.shape} | X_val: {X_val.shape}")


## 3. Khởi tạo & Cấu hình 8 Kịch bản Hồi quy Tweedie (SC-28 đến SC-35)

In [3]:
def create_tweedie_model(model_family: str):
    if model_family == "lightgbm_tweedie":
        return MultiOutputRegressor(
            LGBMRegressor(
                objective="tweedie",
                tweedie_variance_power=1.5,
                learning_rate=0.03,
                num_leaves=31,
                n_estimators=100,
                subsample=0.8,
                colsample_bytree=0.8,
                verbosity=-1,
                random_state=42,
                n_jobs=-1,
            )
        )
    elif model_family == "xgboost_tweedie":
        return MultiOutputRegressor(
            XGBRegressor(
                objective="reg:tweedie",
                tweedie_variance_power=1.5,
                learning_rate=0.03,
                max_depth=5,
                n_estimators=100,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                verbosity=0,
            )
        )
    raise ValueError(f"Unknown family: {model_family}")

scenario_specs = [
    # LightGBM Tweedie (SC-28..31)
    {"scenario_id": "SC-28", "model_family": "lightgbm_tweedie", "feature_set": "F0", "param_count": 7 * 100 * 31},
    {"scenario_id": "SC-29", "model_family": "lightgbm_tweedie", "feature_set": "F1", "param_count": 7 * 100 * 31},
    {"scenario_id": "SC-30", "model_family": "lightgbm_tweedie", "feature_set": "F2", "param_count": 7 * 100 * 31},
    {"scenario_id": "SC-31", "model_family": "lightgbm_tweedie", "feature_set": "F3", "param_count": 7 * 100 * 31},
    # XGBoost Tweedie (SC-32..35)
    {"scenario_id": "SC-32", "model_family": "xgboost_tweedie", "feature_set": "F0", "param_count": 7 * 100 * 32},
    {"scenario_id": "SC-33", "model_family": "xgboost_tweedie", "feature_set": "F1", "param_count": 7 * 100 * 32},
    {"scenario_id": "SC-34", "model_family": "xgboost_tweedie", "feature_set": "F2", "param_count": 7 * 100 * 32},
    {"scenario_id": "SC-35", "model_family": "xgboost_tweedie", "feature_set": "F3", "param_count": 7 * 100 * 32},
]

print(f"Đã khởi tạo {len(scenario_specs)} kịch bản hồi quy Tweedie deviance.")


## 4. Huấn luyện Single-Fit & Lăn Dự báo Đa bước trên Validation

In [4]:
forecasts_dict = {}
benchmark_rows = []
horizon_rows = []

for spec in scenario_specs:
    sc_id = spec["scenario_id"]
    m_fam = spec["model_family"]
    f_set = spec["feature_set"]
    model_name = f"{m_fam}_{f_set}"
    print(f">>> [{sc_id}] Đang huấn luyện & suy luận {model_name}...")

    X_tr = feature_data[f_set]["X_train"]
    Y_tr = feature_data[f_set]["Y_train"]
    X_v = feature_data[f_set]["X_val"]

    model = create_tweedie_model(m_fam)

    # Fit trên 6,293 mẫu Dev Train
    t0 = time.time()
    model.fit(X_tr, Y_tr)
    t_fit = time.time() - t0

    # Suy luận trên 724 Validation origins
    t0 = time.time()
    preds = model.predict(X_v)
    preds = np.clip(preds, 0.0, None)
    t_infer = time.time() - t0

    forecasts_dict[f"{sc_id}_{model_name}"] = preds
    print(f"    Fit: {t_fit:.2f}s | Infer: {t_infer:.2f}s ({(t_infer/num_val_origins)*1000:.2f} ms/origin)")

    # Metrics chuẩn hóa
    row_df = compute_benchmark_metrics(
        y_true=y_val_true,
        y_pred=preds,
        model_name=model_name,
        group="Machine Learning",
        strategy=f"ML Tweedie ({f_set})",
        training_time=t_fit,
        inference_time=t_infer,
        parameter_count=spec["param_count"],
        status=ExecutionStatus.SUCCESS,
        rain_threshold=1.0,
        scale=CANONICAL_DEV_TRAIN_MASE_SCALE,
        horizon=7,
    )
    row_df.insert(0, "scenario_id", sc_id)
    row_df.insert(4, "feature_set", f_set)
    row_df.insert(5, "feature_count", feature_data[f_set]["feature_count"])
    benchmark_rows.append(row_df)

    # Metrics theo từng bước horizon h=1..7
    detailed = evaluate_point_forecast_metrics(y_true=y_val_true, y_pred=preds, scale=CANONICAL_DEV_TRAIN_MASE_SCALE, horizon=7)
    h_rec = {"scenario_id": sc_id, "model": model_name, "feature_set": f_set, "MAE": detailed["MAE"], "RMSE": detailed["RMSE"], "MASE": detailed["MASE"]}
    for h in range(1, 8):
        h_rec[f"MAE@h{h}"] = detailed[f"mae_h{h}"]
        h_rec[f"RMSE@h{h}"] = detailed[f"rmse_h{h}"]
        h_rec[f"MASE@h{h}"] = detailed[f"mase_lag1_h{h}"]
    horizon_rows.append(h_rec)

print("\nHoàn thành 8 kịch bản Machine Learning Tweedie.")


## 5. Bảng Xếp hạng Validation & Lưu Trữ Artifacts

In [5]:
df_results = pd.concat(benchmark_rows, ignore_index=True).sort_values(by="MAE", ascending=True).reset_index(drop=True)
df_horizons = pd.DataFrame(horizon_rows)

# Lưu kết quả vào data/processed/results
results_dir = ROOT_DIR / "data" / "processed" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

val_csv = results_dir / "ml_tweedie_val_results.csv"
val_npz = results_dir / "ml_tweedie_val_forecasts.npz"
h_csv = results_dir / "ml_tweedie_val_horizon_breakdown.csv"

df_results.to_csv(val_csv, index=False)
df_horizons.to_csv(h_csv, index=False)

payload = {"y_true": y_val_true, "origin_dates": origin_dates}
payload.update(forecasts_dict)
np.savez_compressed(val_npz, **payload)

# Tương thích file cũ
df_results.to_csv(results_dir / "ml_tweedie_results.csv", index=False)
np.savez_compressed(results_dir / "ml_tweedie_forecasts.npz", **payload)

print("=== BẢNG XẾP HẠNG 8 KỊCH BẢN ML TWEEDIE (Ranked by MAE) ===")
disp_cols = ['scenario_id', 'model', 'feature_set', 'MAE', 'MSE', 'RMSE', 'MASE', 'training_time', 'inference_time', 'status']
display(df_results[disp_cols])

print("\n=== CHI TIẾT THEO HORIZON (MAE tại h=1..7) ===")
display(df_horizons[['scenario_id', 'model', 'feature_set', 'MAE'] + [f'MAE@h{h}' for h in range(1, 8)]])
